# How much fine-tuning is worth (edited 2026-08-25)

Three arms at sixteen folds on the current dataset: the base frozen, the last
VGG19 block released, and the last two released.

| arm | trunk | where |
|---|---|---|
| `frozen` | base frozen, features cached | already run locally |
| `block4` | block4 released, 5 epochs at 1e-5 | here |
| `block34` | block4 and block3 released, 5 epochs at 1e-5 | here |

**The frozen arm is not run here.** It trains on cached features, which is a
hundred minutes on a CPU, and a first attempt spent a GPU session on it and then
lost thirteen folds when the runtime died. The GPU is for the two arms that
cannot run locally. Sync is now per fold rather than per arm, so a dead session
costs the fold in flight and nothing before it, and rerunning resumes.

**Read the existing three-fold answer with care.** It says frozen 0.6992,
block4 0.9416, block34 0.9790, and per station it says 0.959 / 0.965 / 0.987 at
IPA13ST and 0.969 / 0.971 / 0.970 at IPA20ST. That is no difference. The whole
spread comes from IPA4ST, where the frozen arm scores 0.1693 -- 100 calls in
2,470 detections is a 4.0 % base rate, and at that base rate precision is a
knife edge that a fitted threshold can land on either side of.

So this runs sixteen folds, reports paired differences per station rather than a
macro mean, and prints every comparison a second time with IPA4ST removed. The
scan that follows is what actually decides it: a scan does not depend on where a
threshold falls inside a fixed low-base-rate set.

Unfreezing reads the image pack rather than the feature cache, so it is roughly
eight times slower per epoch and wants the GPU.

**The three folds this notebook produced on 24 August are being redone, and it
is worth knowing why.** They have 24 columns where a local fold has 32.
gated_detections is 0 and every gated_loso_ column is simply absent -- the four
the paper quotes and the four the comparison below reads. The time gate reads
its clock from cleanup_vs_review.csv, that file carries the recorders'
coordinates and so never goes on Drive, and without it train_v13_loso.py prints
one line and writes the CSV anyway. Twenty-five GPU-minutes each, and the run
looked like progress.

Fixed two ways. The gate now has its own table -- filename, timestamp, offset,
no verdicts and no coordinates -- which is committed to the repository, so the
clone below carries it and nothing needs uploading. And the skip rule now checks
a synced fold for those columns instead of only checking that it exists, so the
three bad ones are redone rather than protected.

**The same two arms are also running locally**, working backwards from IPA20ST
while this works forwards from IPA1ST. They cannot see each other -- this writes
to Drive, that writes to data/outputs -- and running in the same direction cost
this machine 19.7 hours retraining a fold that had already finished here. From
opposite ends they converge instead. Run this whenever the GPU is available and
let the local copy come the other way.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!rm -rf /content/repo && git clone -q -b v13-honest-labels https://github.com/Mo119m/primates-sound-detection /content/repo
%cd /content/repo
!git log --oneline -1
# The three files landed at the top of primates-sound-detection rather than in
# a subfolder. Copying them by name from there rather than asking for the upload
# to be redone; the next cell checks the row count, which is what actually
# distinguishes this dataset from the one beside it.
U = '/content/drive/MyDrive/primates-sound-detection'
!mkdir -p /content/dataF && cp -n {U}/v13_images.npy {U}/v13_index.csv {U}/manifest.csv /content/dataF/
!ls -la /content/dataF

In [ ]:
# The uploaded copy is already the corrected one, built 2026-08-19 with the
# 27 field pogonias and the 1,056 expert bird clips in it. Verify rather than
# assume: 22,169 rows is this dataset, 21,120 is the one before it.
import pandas as pd
i = pd.read_csv('/content/dataF/v13_index.csv')
print(len(i), 'rows')
assert len(i) == 22169, 'wrong dataset on Drive'
print(i.label.value_counts().to_dict())

In [ ]:
# The time gate needs its lookup table. Without it every fold trains
# fine and comes out missing the four columns the comparison reads, which is
# what happened on 24 August. Fail here instead, where it costs nothing.
import os
import pandas as pd
G = '/content/repo/data/outputs/auto_cleanup/review_gate_table.csv'
assert os.path.exists(G), 'no review gate table in the clone -- pull the branch again'
g = pd.read_csv(G)
print(len(g), 'rows,', list(g.columns))
assert len(g) == 6189, 'wrong gate table'
assert set(g.columns) == {'file', 'timestamp', 'start_s'}, 'unexpected columns'
print('time gate has its clock')

In [ ]:
A = '--manifest /content/dataF/manifest.csv --index /content/dataF/v13_index.csv --images /content/dataF/v13_images.npy --cache /content/dataF/v13_features.npy'
!python scripts/train_v13_loso.py --prepare-cache-only --overwrite {A} --out /content/warm.csv --run-metadata /content/cacheF.run.json

In [ ]:
# Two unfrozen arms, sixteen folds each, synced fold by fold.
# Safe to re-run after a disconnect: folds already in Drive are skipped.
!python colab/run_unfreeze_arms.py

In [ ]:
# Read the arms back without retraining, once they exist.
import sys
sys.argv = ['x']
sys.path.insert(0, '/content/repo/colab')
import run_unfreeze_arms
run_unfreeze_arms.summarise()